<a href="https://colab.research.google.com/github/Guettanani/site_parfum_gallery/blob/master/Copie_de_Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Revenir à la racine de Colab et supprimer l'ancienne tentative
%cd /content
!rm -rf YOLOX

# 2. Cloner à nouveau le dépôt YOLOX proprement
!git clone https://github.com/Megvii-BaseDetection/YOLOX.git
%cd YOLOX

# 3. Corriger le fichier requirements.txt (remplace l'ancienne version par onnxsim compatible Python 3.12)
!sed -i 's/onnx-simplifier==0.4.10/onnxsim/g' requirements.txt

# 4. Installer les dépendances mises à jour
!pip3 install -r requirements.txt

# 5. Installer YOLOX
!pip3 install --no-build-isolation -v -e .

/content
Cloning into 'YOLOX'...
remote: Enumerating objects: 1940, done.
remote: Counting objects: 100% (203/203), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 1940 (delta 172), reused 164 (delta 164), pack-reused 1737 (from 1)
Receiving objects: 100% (1940/1940), 7.55 MiB | 19.66 MiB/s, done.
Resolving deltas: 100% (1171/1171), done.
/content/YOLOX
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 115.1 MB/s eta 0:00:00
Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
Obtaining file:///content/YOLOX
  Running command python setup.py egg_info
  /usr/local/lib/python3.12/dist-packages/setuptools/__init__.py:94: _DeprecatedInstaller: setuptools.installer and fetch_build_eggs are deprecated.
  

In [ ]:
import os, sys
from google.colab import drive

# Montage de Google Drive pour accéder au dataset
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Mettre à jour pip, setuptools et wheel pour éviter les erreurs de construction
!pip install -q --upgrade pip setuptools wheel

# Installation des dépendances (versions corrigées pour N6), ajout de cython
!pip install -q pycocotools loguru wandb albumentations cython

# CORRECTIF 1 : Retrait de la version figée "==0.4.36" pour utiliser une version compatible Python 3.12 pré-compilée
!pip install -q onnx==1.16.0 onnxsim "onnxruntime>=1.17.0"

# Installation de YOLOX (Correctif build-isolation)
if not os.path.exists('YOLOX'):
    !git clone -q https://github.com/Megvii-BaseDetection/YOLOX.git

    # Patch setup.py pour éviter l'erreur fetch_build_eggs
    !sed -i 's/setup_requires=\["wheel"\]/setup_requires=[]/g' YOLOX/setup.py
    !sed -i 's/setup_requires=\["numpy>=1.14.0"\]/setup_requires=[]/g' YOLOX/setup.py

    # CORRECTIF 2 : Remplacer onnx-simplifier obsolète par onnxsim compatible Python 3.12
    !sed -i 's/onnx-simplifier==0.4.10/onnxsim/g' YOLOX/requirements.txt

    # Lancement de l'installation de YOLOX
    !cd YOLOX && pip install -q -e . --no-build-isolation

sys.path.insert(0, os.getcwd() + '/YOLOX')
print('✓ Setup OK')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
  Checking if build backend supports build_editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for yolox (pyproject.toml) ... done
✓ Setup OK


In [ ]:
# ==========================================================
# 1. MONTAGE DU DRIVE (pour sauvegarder uniquement les annotations)
# ==========================================================
from google.colab import drive, files
drive.mount('/content/drive')

# ==========================================================
# 2. CONFIGURATION DE L'API KAGGLE
# ==========================================================
!pip install -q kaggle
# Téléversez votre fichier "kaggle.json" ici
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# ==========================================================
# 3. TÉLÉCHARGEMENT & DÉCOMPRESSION LOCALE (Très rapide)
# ==========================================================
# Création du dossier local dans la machine Colab (pas sur le Drive)
!mkdir -p /content/bdd100k_local

print("Téléchargement du dataset brut sur le disque local de Colab...")
!kaggle datasets download -d solesensei/solesensei_bdd100k -p /content/bdd100k_local

print("Décompression locale (cette opération prend moins de 2 minutes en local)...")
!unzip -q /content/bdd100k_local/*.zip -d /content/bdd100k_local/

# ==========================================================
# 4. CONVERSION AUTOMATIQUE EN COCO (Au niveau de la source)
# ==========================================================
print("\nConversion des annotations BDD en format COCO JSON...")

import os
import json
from tqdm import tqdm

# Chemins locaux sur la machine Colab
src_train_json = "/content/bdd100k_local/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_train.json"
src_val_json = "/content/bdd100k_local/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_val.json"

# Chemins de destination (Sur le disque local ET sur votre Drive pour sauvegarde)
local_anno_dir = "/content/bdd100k_local/annotations"
drive_anno_dir = "/content/drive/MyDrive/datasets/bdd100k/annotations"

os.makedirs(local_anno_dir, exist_ok=True)
os.makedirs(drive_anno_dir, exist_ok=True)

dst_train_coco = os.path.join(local_anno_dir, "instances_train2017.json")
dst_val_coco = os.path.join(local_anno_dir, "instances_val2017.json")

# Classes cibles officielles
categories = [
    {"supercategory": "none", "id": 1, "name": "person"},
    {"supercategory": "none", "id": 2, "name": "rider"},
    {"supercategory": "none", "id": 3, "name": "car"},
    {"supercategory": "none", "id": 4, "name": "bus"},
    {"supercategory": "none", "id": 5, "name": "truck"},
    {"supercategory": "none", "id": 6, "name": "bike"},
    {"supercategory": "none", "id": 7, "name": "motor"},
    {"supercategory": "none", "id": 8, "name": "traffic light"},
    {"supercategory": "none", "id": 9, "name": "traffic sign"},
    {"supercategory": "none", "id": 10, "name": "train"},
]
cat_to_id = {cat["name"]: cat["id"] for cat in categories}

def convert_bdd_to_coco(src_path, dst_path):
    if not os.path.exists(src_path):
        print(f"Fichier introuvable : {src_path}")
        return

    with open(src_path, "r") as f:
        bdd_data = json.load(f)

    coco_data = {
        "images": [],
        "annotations": [],
        "categories": categories
    }

    image_id_counter = 1
    ann_id_counter = 1

    for item in tqdm(bdd_data, desc="Conversion en cours"):
        image_info = {
            "file_name": item["name"],
            "height": 720,
            "width": 1280,
            "id": image_id_counter
        }

        has_valid_ann = False

        if "labels" in item and item["labels"] is not None:
            for label in item["labels"]:
                category_name = label.get("category")
                if category_name in cat_to_id and "box2d" in label:
                    has_valid_ann = True
                    box = label["box2d"]
                    x1 = box["x1"]
                    y1 = box["y1"]
                    x2 = box["x2"]
                    y2 = box["y2"]

                    width = x2 - x1
                    height = y2 - y1

                    ann_info = {
                        "id": ann_id_counter,
                        "image_id": image_id_counter,
                        "category_id": cat_to_id[category_name],
                        "bbox": [x1, y1, width, height],
                        "area": float(width * height),
                        "iscrowd": 0
                    }
                    coco_data["annotations"].append(ann_info)
                    ann_id_counter += 1

        if has_valid_ann:
            coco_data["images"].append(image_info)
            image_id_counter += 1

    with open(dst_path, "w") as f:
        json.dump(coco_data, f)
    print(f"Fichier exporté avec succès : {dst_path}")

# Exécution de la conversion
convert_bdd_to_coco(src_train_json, dst_train_coco)
convert_bdd_to_coco(src_val_json, dst_val_coco)

# ==========================================================
# 5. SAUVEGARDE DES LABELS SUR VOTRE GOOGLE DRIVE
# ==========================================================
print("\nSauvegarde des nouveaux fichiers COCO JSON sur votre Google Drive...")
!cp /content/bdd100k_local/annotations/*.json /content/drive/MyDrive/datasets/bdd100k/annotations/
print("Sauvegarde terminée !")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving kaggle.json to kaggle.json
Téléchargement du dataset brut sur le disque local de Colab...
Dataset URL: https://www.kaggle.com/datasets/solesensei/solesensei_bdd100k
License(s): other
100% 7.61G/7.61G [01:27<00:00, 93.3MB/s]

Décompression locale (cette opération prend moins de 2 minutes en local)...

Conversion des annotations BDD en format COCO JSON...


Conversion en cours: 100%|██████████| 69863/69863 [00:11<00:00, 6289.43it/s] 


Fichier exporté avec succès : /content/bdd100k_local/annotations/instances_train2017.json


Conversion en cours: 100%|██████████| 10000/10000 [00:01<00:00, 8324.03it/s]


Fichier exporté avec succès : /content/bdd100k_local/annotations/instances_val2017.json

Sauvegarde des nouveaux fichiers COCO JSON sur votre Google Drive...
Sauvegarde terminée !


In [ ]:
# Vérifier la taille du dossier local (doit faire plusieurs gigaoctets)
!du -sh /content/bdd100k_local/

# Afficher les 10 premières images d'entraînement pour confirmer
!ls /content/bdd100k_local/bdd100k/images/100k/train | head -n 10

18G	/content/bdd100k_local/
ls: cannot access '/content/bdd100k_local/bdd100k/images/100k/train': No such file or directory


In [ ]:
# 1. Créer le dossier de destination "images" sur votre Google Drive
!mkdir -p /content/drive/MyDrive/datasets/bdd100k/images/

# 2. Copier le dossier "100k" (qui contient train et val) vers votre Google Drive
!cp -r /content/bdd100k_local/bdd100k/bdd100k/images/100k /content/drive/MyDrive/datasets/bdd100k/images/

In [ ]:
import os
import json

# ==========================================================
# 1. DÉTECTION DÉGAGEANT LES CHEMINS D'ACCÈS
# ==========================================================
print("Recherche des sources de données...")

# Détection de la source des images (Local vs Drive)
local_img_train = "/content/bdd100k_local/bdd100k/bdd100k/images/100k/train"
local_img_val = "/content/bdd100k_local/bdd100k/bdd100k/images/100k/val"

drive_img_train = "/content/drive/MyDrive/datasets/bdd100k/images/100k/train"
drive_img_val = "/content/drive/MyDrive/datasets/bdd100k/images/100k/val"

if os.path.exists(local_img_train):
    img_train_src = local_img_train
    img_val_src = local_img_val
    print("✓ Utilisation détectée : Images locales rapides de la VM Colab.")
elif os.path.exists(drive_img_train):
    img_train_src = drive_img_train
    img_val_src = drive_img_val
    print("✓ Utilisation détectée : Images stockées sur votre Google Drive.")
else:
    raise FileNotFoundError("Erreur : Impossible de localiser le dossier contenant les images BDD100k.")

# Détection de la source des annotations COCO JSON
local_anno_train = "/content/bdd100k_local/annotations/instances_train2017.json"
local_anno_val = "/content/bdd100k_local/annotations/instances_val2017.json"

drive_anno_train = "/content/drive/MyDrive/datasets/bdd100k/annotations/instances_train2017.json"
drive_anno_val = "/content/drive/MyDrive/datasets/bdd100k/annotations/instances_val2017.json"

if os.path.exists(local_anno_train):
    anno_train_src = local_anno_train
    anno_val_src = local_anno_val
    print("✓ Utilisation détectée : Annotations JSON locales.")
elif os.path.exists(drive_anno_train):
    anno_train_src = drive_anno_train
    anno_val_src = drive_anno_val
    print("✓ Utilisation détectée : Annotations JSON sur votre Google Drive.")
else:
    raise FileNotFoundError("Erreur : Impossible de trouver les fichiers COCO JSON convertis.")

# ==========================================================
# 2. STRUCTURATION AU FORMAT COCO STANDARD
# ==========================================================
print("\nRestructuration du dossier d'entraînement...")
target_dir = "/content/bdd100k_coco"

# Nettoyer l'ancienne arborescence si nécessaire
!rm -rf {target_dir}
os.makedirs(os.path.join(target_dir, "annotations"), exist_ok=True)

# Création des liens symboliques pour une restructuration instantanée
os.symlink(img_train_src, os.path.join(target_dir, "train2017"))
os.symlink(img_val_src, os.path.join(target_dir, "val2017"))
os.symlink(anno_train_src, os.path.join(target_dir, "annotations/instances_train2017.json"))
os.symlink(anno_val_src, os.path.join(target_dir, "annotations/instances_val2017.json"))

print(f"✓ Arborescence créée avec succès dans : {target_dir}")

# ==========================================================
# 3. COMPATIBILITÉ ET VALIDATION DU DATASET
# ==========================================================
def validate_dataset(img_dir, json_path, dataset_name):
    print(f"\n--- Diagnostic du dataset : {dataset_name} ---")

    # 1. Tentative de lecture du fichier d'annotations
    try:
        with open(json_path, "r") as f:
            data = json.load(f)
        print("  [✓] Lecture du fichier JSON d'annotations réussie.")
    except Exception as e:
        print(f"  [X] Échec de la lecture du fichier JSON : {e}")
        return False

    # 2. Analyse des données chargées
    num_images = len(data.get("images", []))
    num_annotations = len(data.get("annotations", []))
    num_categories = len(data.get("categories", []))

    print(f"  [✓] Données déclarées : {num_images} images, {num_annotations} objets, {num_categories} classes.")

    if num_images == 0 or num_annotations == 0:
        print("  [X] Erreur : Le fichier d'annotations ne contient pas d'images ou d'objets valides.")
        return False

    # 3. Vérification de l'alignement des fichiers d'images sur le disque
    missing_images = 0
    checked_limit = min(200, num_images) # On teste l'existence des 200 premières images

    for i in range(checked_limit):
        file_name = data["images"][i]["file_name"]
        full_path = os.path.join(img_dir, file_name)
        if not os.path.exists(full_path):
            missing_images += 1
            if missing_images <= 3:
                print(f"  [X] Image déclarée absente sur le disque : {full_path}")

    if missing_images == 0:
        print(f"  [✓] Intégrité disque : Les images tests existent bien sur le système.")
        return True
    else:
        print(f"  [X] Erreur d'intégrité : {missing_images} images introuvables sur les {checked_limit} vérifiées.")
        return False

# Lancement de la validation
train_ok = validate_dataset(os.path.join(target_dir, "train2017"), os.path.join(target_dir, "annotations/instances_train2017.json"), "ENTRAÎNEMENT")
val_ok = validate_dataset(os.path.join(target_dir, "val2017"), os.path.join(target_dir, "annotations/instances_val2017.json"), "VALIDATION")

if train_ok and val_ok:
    print("\n=======================================================")
    print("✓ DIAGNOSTIC POSITIF : La base de données est prête !")
    print("Structure configurée et entièrement compatible avec YOLOX.")
    print("=======================================================")
else:
    print("\n=======================================================")
    print("⚠ DIAGNOSTIC NÉGATIF : Un problème a été identifié.")
    print("Veuillez corriger les erreurs de chemins d'accès ci-dessus.")
    print("=======================================================")

Recherche des sources de données...
✓ Utilisation détectée : Images locales rapides de la VM Colab.
✓ Utilisation détectée : Annotations JSON locales.

Restructuration du dossier d'entraînement...
✓ Arborescence créée avec succès dans : /content/bdd100k_coco

--- Diagnostic du dataset : ENTRAÎNEMENT ---
  [✓] Lecture du fichier JSON d'annotations réussie.
  [✓] Données déclarées : 69863 images, 1286871 objets, 10 classes.
  [X] Image déclarée absente sur le disque : /content/bdd100k_coco/train2017/0000f77c-6257be58.jpg
  [X] Image déclarée absente sur le disque : /content/bdd100k_coco/train2017/0000f77c-62c2a288.jpg
  [X] Image déclarée absente sur le disque : /content/bdd100k_coco/train2017/0000f77c-cb820c98.jpg
  [X] Erreur d'intégrité : 196 images introuvables sur les 200 vérifiées.

--- Diagnostic du dataset : VALIDATION ---
  [✓] Lecture du fichier JSON d'annotations réussie.
  [✓] Données déclarées : 10000 images, 185526 objets, 10 classes.
  [✓] Intégrité disque : Les images tes

In [ ]:
import os
import json
import glob
from tqdm import tqdm

# ==========================================================
# 1. DÉTECTION DES SOURCES DE DONNÉES
# ==========================================================
print("Recherche des sources de données...")

local_img_train = "/content/bdd100k_local/bdd100k/bdd100k/images/100k/train"
local_img_val = "/content/bdd100k_local/bdd100k/bdd100k/images/100k/val"

drive_img_train = "/content/drive/MyDrive/datasets/bdd100k/images/100k/train"
drive_img_val = "/content/drive/MyDrive/datasets/bdd100k/images/100k/val"

if os.path.exists(local_img_train):
    img_train_src = local_img_train
    img_val_src = local_img_val
    print("✓ Utilisation détectée : Images locales rapides de la VM Colab.")
elif os.path.exists(drive_img_train):
    img_train_src = drive_img_train
    img_val_src = drive_img_val
    print("✓ Utilisation détectée : Images stockées sur votre Google Drive.")
else:
    raise FileNotFoundError("Erreur : Impossible de localiser le dossier contenant les images BDD100k.")

# Détection des annotations
local_anno_train = "/content/bdd100k_local/annotations/instances_train2017.json"
local_anno_val = "/content/bdd100k_local/annotations/instances_val2017.json"
drive_anno_train = "/content/drive/MyDrive/datasets/bdd100k/annotations/instances_train2017.json"
drive_anno_val = "/content/drive/MyDrive/datasets/bdd100k/annotations/instances_val2017.json"

if os.path.exists(local_anno_train):
    anno_train_src = local_anno_train
    anno_val_src = local_anno_val
    print("✓ Utilisation détectée : Annotations JSON locales.")
elif os.path.exists(drive_anno_train):
    anno_train_src = drive_anno_train
    anno_val_src = drive_anno_val
    print("✓ Utilisation détectée : Annotations JSON sur votre Google Drive.")
else:
    raise FileNotFoundError("Erreur : Fichiers COCO JSON introuvables.")

# ==========================================================
# 2. STRUCTURATION ET MISE À PLAT DU DATASET (CORRECTION)
# ==========================================================
print("\nRestructuration chirurgicale du dossier d'entraînement...")
target_dir = "/content/bdd100k_coco"

# Nettoyer l'ancienne arborescence
!rm -rf {target_dir}
os.makedirs(os.path.join(target_dir, "annotations"), exist_ok=True)
os.makedirs(os.path.join(target_dir, "train2017"), exist_ok=True)
os.makedirs(os.path.join(target_dir, "val2017"), exist_ok=True)

# Liaison directe des fichiers d'annotations
os.symlink(anno_train_src, os.path.join(target_dir, "annotations/instances_train2017.json"))
os.symlink(anno_val_src, os.path.join(target_dir, "annotations/instances_val2017.json"))

# 2a. Scan récursif pour aplatir le set d'Entraînement (trainA, trainB, testA, testB)
print("Scanning de toutes les images d'entraînement (y compris dans les sous-dossiers cachés)...")
train_jpgs = glob.glob(os.path.join(img_train_src, "**/*.jpg"), recursive=True)

print(f"Liaison de {len(train_jpgs)} images d'entraînement vers train2017...")
for filepath in tqdm(train_jpgs, desc="Création des liens d'entraînement"):
    filename = os.path.basename(filepath)
    linkpath = os.path.join(target_dir, "train2017", filename)
    if not os.path.exists(linkpath):
        os.symlink(filepath, linkpath)

# 2b. Scan récursif pour le set de Validation (par sécurité)
val_jpgs = glob.glob(os.path.join(img_val_src, "**/*.jpg"), recursive=True)
print(f"Liaison de {len(val_jpgs)} images de validation vers val2017...")
for filepath in tqdm(val_jpgs, desc="Création des liens de validation"):
    filename = os.path.basename(filepath)
    linkpath = os.path.join(target_dir, "val2017", filename)
    if not os.path.exists(linkpath):
        os.symlink(filepath, linkpath)

print(f"\n✓ Arborescence mise à plat créée avec succès dans : {target_dir}")

# ==========================================================
# 3. NOUVELLE VALIDATION DIAGNOSTIQUE
# ==========================================================
def validate_dataset(img_dir, json_path, dataset_name):
    print(f"\n--- Diagnostic du dataset : {dataset_name} ---")

    try:
        with open(json_path, "r") as f:
            data = json.load(f)
        print("  [✓] Lecture du fichier JSON d'annotations réussie.")
    except Exception as e:
        print(f"  [X] Échec de la lecture du fichier JSON : {e}")
        return False

    num_images = len(data.get("images", []))
    num_annotations = len(data.get("annotations", []))
    num_categories = len(data.get("categories", []))

    print(f"  [✓] Données déclarées : {num_images} images, {num_annotations} objets, {num_categories} classes.")

    if num_images == 0 or num_annotations == 0:
        print("  [X] Erreur : Le fichier d'annotations ne contient pas d'images ou d'objets valides.")
        return False

    missing_images = 0
    checked_limit = min(200, num_images)

    for i in range(checked_limit):
        file_name = data["images"][i]["file_name"]
        full_path = os.path.join(img_dir, file_name)
        if not os.path.exists(full_path):
            missing_images += 1
            if missing_images <= 3:
                print(f"  [X] Image déclarée absente sur le disque : {full_path}")

    if missing_images == 0:
        print(f"  [✓] Intégrité disque : Les images tests ont été localisées avec succès.")
        return True
    else:
        print(f"  [X] Erreur d'intégrité : {missing_images} images manquantes parmi les {checked_limit} vérifiées.")
        return False

# Lancement du diagnostic de contrôle
train_ok = validate_dataset(os.path.join(target_dir, "train2017"), os.path.join(target_dir, "annotations/instances_train2017.json"), "ENTRAÎNEMENT")
val_ok = validate_dataset(os.path.join(target_dir, "val2017"), os.path.join(target_dir, "annotations/instances_val2017.json"), "VALIDATION")

if train_ok and val_ok:
    print("\n=======================================================")
    print("✓ DIAGNOSTIC CONFIRMÉ : Le dataset est 100% opérationnel !")
    print("Toutes les images d'entraînement ont été reliées.")
    print("YOLOX peut maintenant être entraîné sans aucune erreur d'I/O.")
    print("=======================================================")
else:
    print("\n=======================================================")
    print("⚠ ÉCHEC DU DIAGNOSTIC : Des incohérences subsistent.")
    print("=======================================================")

Recherche des sources de données...
✓ Utilisation détectée : Images locales rapides de la VM Colab.
✓ Utilisation détectée : Annotations JSON locales.

Restructuration chirurgicale du dossier d'entraînement...
Scanning de toutes les images d'entraînement (y compris dans les sous-dossiers cachés)...
Liaison de 70000 images d'entraînement vers train2017...


Création des liens d'entraînement: 100%|██████████| 70000/70000 [00:07<00:00, 8985.59it/s] 


Liaison de 10000 images de validation vers val2017...


Création des liens de validation: 100%|██████████| 10000/10000 [00:00<00:00, 13249.60it/s]



✓ Arborescence mise à plat créée avec succès dans : /content/bdd100k_coco

--- Diagnostic du dataset : ENTRAÎNEMENT ---
  [✓] Lecture du fichier JSON d'annotations réussie.
  [✓] Données déclarées : 69863 images, 1286871 objets, 10 classes.
  [✓] Intégrité disque : Les images tests ont été localisées avec succès.

--- Diagnostic du dataset : VALIDATION ---
  [✓] Lecture du fichier JSON d'annotations réussie.
  [✓] Données déclarées : 10000 images, 185526 objets, 10 classes.
  [✓] Intégrité disque : Les images tests ont été localisées avec succès.

✓ DIAGNOSTIC CONFIRMÉ : Le dataset est 100% opérationnel !
Toutes les images d'entraînement ont été reliées.
YOLOX peut maintenant être entraîné sans aucune erreur d'I/O.


In [ ]:
import os

# 1. Vérification de sécurité
if not os.path.exists("/content/bdd100k_coco/train2017") or not os.path.exists("/content/bdd100k_coco/annotations"):
    raise FileNotFoundError("Erreur : Veuillez d'abord exécuter la cellule précédente de restructuration pour générer l'arborescence locale.")

print("Début de la création de l'archive ZIP durable...")
print("Les liens symboliques vont être automatiquement résolus pour inclure physiquement toutes vos images.")
print("Cette opération peut prendre entre 5 et 10 minutes car elle traite 8 Go d'images. Veuillez patienter...")

# 2. Compression du dossier 'bdd100k_coco' directement vers le dossier de votre Google Drive
# L'absence de l'option '-y' force 'zip' à résoudre les liens physiques et à copier les images réelles.
!cd /content && zip -q -r /content/drive/MyDrive/datasets/bdd100k_coco.zip bdd100k_coco

print("\n=======================================================")
print("✓ ARCHIVAGE DURABLE TERMINÉ AVEC SUCCÈS !")
print("Votre dataset BDD100K au format COCO à plat est sauvegardé sur votre Drive :")
print("-> /content/drive/MyDrive/datasets/bdd100k_coco.zip")
print("=======================================================")

Début de la création de l'archive ZIP durable...
Les liens symboliques vont être automatiquement résolus pour inclure physiquement toutes vos images.
Cette opération peut prendre entre 5 et 10 minutes car elle traite 8 Go d'images. Veuillez patienter...

✓ ARCHIVAGE DURABLE TERMINÉ AVEC SUCCÈS !
Votre dataset BDD100K au format COCO à plat est sauvegardé sur votre Drive :
-> /content/drive/MyDrive/datasets/bdd100k_coco.zip


In [ ]:
# 1. Monter le Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Copier l'archive ultra-rapidement sur le disque local rapide du Colab actuel
!cp /content/drive/MyDrive/datasets/bdd100k_coco.zip /content/

# 3. Décompresser le dataset (prend environ 1 à 2 minutes)
!unzip -q /content/bdd100k_coco.zip -d /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
replace /content/bdd100k_coco/annotations/instances_train2017.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/bdd100k_coco/annotations/instances_val2017.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


In [ ]:
# Lister le contenu du dossier de votre Drive avec sa taille réelle
!ls -lh /content/drive/MyDrive/datasets/

total 4.3G
drwx------ 4 root root 4.0K May 25 18:36 bdd100k
-rw------- 1 root root 4.3G May 25 19:27 bdd100k_coco.zip


In [ ]:
# 1. Montage du Drive
from google.colab import drive, files
import os
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Copie locale du dataset structuré (rapide)
!cp /content/drive/MyDrive/datasets/bdd100k_coco.zip /content/

# 3. Décompression locale
print("Décompression du dataset d'entraînement...")
!unzip -q /content/bdd100k_coco.zip -d /content/
print("✓ Dataset prêt à l'emplacement : /content/bdd100k_coco")

cp: cannot stat '/content/drive/MyDrive/datasets/bdd100k_coco.zip': No such file or directory
Décompression du dataset d'entraînement...
unzip:  cannot find or open /content/bdd100k_coco.zip, /content/bdd100k_coco.zip.zip or /content/bdd100k_coco.zip.ZIP.
✓ Dataset prêt à l'emplacement : /content/bdd100k_coco


In [ ]:
config_content = """
import os
from yolox.exp import Exp as MyExp

class Exp(MyExp):
    def __init__(self):
        super(Exp, self).__init__()
        # Paramètres d'échelle de YOLOX-Nano
        self.depth = 0.33
        self.width = 0.25

        # Résolution d'entrée robuste pour STM32N6
        self.input_size = (416, 416)
        self.test_size = (416, 416)

        # Paramètres d'apprentissage
        self.num_classes = 10           # Les 10 classes de BDD100k
        self.max_epoch = 80             # Limite pour éviter le sur-apprentissage
        self.data_num_workers = 4
        self.eval_interval = 5          # Évaluation toutes les 5 époques
        self.exp_name = "yolox_nano_bdd100k"
        self.multiscale_range = 0       # Désactiver pour figer la dimension mémoire

    def get_dataset(self, cache: bool = False, cache_type: str = "ram"):
        from yolox.data import COCODataset
        from yolox.data import TrainTransform

        return COCODataset(
            data_dir="/content/bdd100k_coco",
            json_file="instances_train2017.json",
            name="train2017",
            img_size=self.input_size,
            preproc=TrainTransform(
                max_labels=50,
                flip_prob=0.5,
                hsv_prob=1.0
            ),
            cache=cache,
            cache_type=cache_type,
        )

    def get_eval_dataset(self, **kwargs):
        from yolox.data import COCODataset
        from yolox.data import ValTransform

        return COCODataset(
            data_dir="/content/bdd100k_coco",
            json_file="instances_val2017.json",
            name="val2017",
            img_size=self.test_size,
            preproc=ValTransform(legacy=False),
        )
"""

# Écriture du fichier dans le répertoire de YOLOX
os.makedirs("YOLOX/exps/custom", exist_ok=True)
with open("YOLOX/exps/custom/yolox_nano_custom.py", "w") as f:
    f.write(config_content)
print("✓ Fichier d'expérience YOLOX écrit avec succès.")

✓ Fichier d'expérience YOLOX écrit avec succès.


In [ ]:
import os
import glob
from tqdm import tqdm

def heal_and_reconstruct():
    target_dir = "/content/bdd100k_coco"

    # 1. Vérification de l'intégrité existante
    target_json = os.path.join(target_dir, "annotations/instances_train2017.json")
    if os.path.exists(target_json) and os.path.getsize(target_json) > 0:
        print("✓ Le dataset bdd100k_coco est déjà présent, complet et valide.")
        return

    print("⚠ Le dataset bdd100k_coco est manquant ou incomplet.")
    print("Lancement de la reconstruction et de la mise à plat automatique...")

    # 2. Détection de la source des images (Local VM vs Google Drive)
    local_img_train = "/content/bdd100k_local/bdd100k/bdd100k/images/100k/train"
    local_img_val = "/content/bdd100k_local/bdd100k/bdd100k/images/100k/val"
    drive_img_train = "/content/drive/MyDrive/datasets/bdd100k/images/100k/train"
    drive_img_val = "/content/drive/MyDrive/datasets/bdd100k/images/100k/val"

    if os.path.exists(local_img_train):
        img_train_src = local_img_train
        img_val_src = local_img_val
        print("  -> Source d'images détectée : Disque local temporaire rapide de la VM.")
    elif os.path.exists(drive_img_train):
        img_train_src = drive_img_train
        img_val_src = drive_img_val
        print("  -> Source d'images détectée : Google Drive (Vitesse d'accès réseau).")
    else:
        raise FileNotFoundError("Erreur : Impossible de trouver vos dossiers d'images BDD100k, ni en local, ni sur votre Drive.")

    # Détection de la source des annotations COCO JSON
    local_anno_train = "/content/bdd100k_local/annotations/instances_train2017.json"
    local_anno_val = "/content/bdd100k_local/annotations/instances_val2017.json"
    drive_anno_train = "/content/drive/MyDrive/datasets/bdd100k/annotations/instances_train2017.json"
    drive_anno_val = "/content/drive/MyDrive/datasets/bdd100k/annotations/instances_val2017.json"

    if os.path.exists(local_anno_train):
        anno_train_src = local_anno_train
        anno_val_src = local_anno_val
        print("  -> Source d'annotations : Fichiers JSON locaux.")
    elif os.path.exists(drive_anno_train):
        anno_train_src = drive_anno_train
        anno_val_src = drive_anno_val
        print("  -> Source d'annotations : Fichiers JSON sur Google Drive.")
    else:
        raise FileNotFoundError("Erreur : Impossible de trouver vos annotations converties, ni en local, ni sur votre Drive.")

    # 3. Nettoyage et reconstruction propre de la structure
    !rm -rf {target_dir}
    os.makedirs(os.path.join(target_dir, "annotations"), exist_ok=True)
    os.makedirs(os.path.join(target_dir, "train2017"), exist_ok=True)
    os.makedirs(os.path.join(target_dir, "val2017"), exist_ok=True)

    # Création des liens symboliques pour les JSON
    os.symlink(anno_train_src, os.path.join(target_dir, "annotations/instances_train2017.json"))
    os.symlink(anno_val_src, os.path.join(target_dir, "annotations/instances_val2017.json"))

    # Liaison à plat des images d'entraînement (résout l'imbrication trainA/B, etc.)
    print("  -> Liaison et mise à plat des images d'entraînement...")
    train_jpgs = glob.glob(os.path.join(img_train_src, "**/*.jpg"), recursive=True)
    for filepath in tqdm(train_jpgs, desc="Mise à plat (Train)"):
        filename = os.path.basename(filepath)
        os.symlink(filepath, os.path.join(target_dir, "train2017", filename))

    # Liaison à plat des images de validation
    print("  -> Liaison et mise à plat des images de validation...")
    val_jpgs = glob.glob(os.path.join(img_val_src, "**/*.jpg"), recursive=True)
    for filepath in tqdm(val_jpgs, desc="Mise à plat (Val)"):
        filename = os.path.basename(filepath)
        os.symlink(filepath, os.path.join(target_dir, "val2017", filename))

    print("✓ RECONSTRUCTION ET CORRECTION TERMINÉES !")

heal_and_reconstruct()

⚠ Le dataset bdd100k_coco est manquant ou incomplet.
Lancement de la reconstruction et de la mise à plat automatique...
  -> Source d'images détectée : Google Drive (Vitesse d'accès réseau).
  -> Source d'annotations : Fichiers JSON sur Google Drive.
  -> Liaison et mise à plat des images d'entraînement...


Mise à plat (Train): 100%|██████████| 62538/62538 [00:02<00:00, 23739.44it/s]


  -> Liaison et mise à plat des images de validation...


Mise à plat (Val): 0it [00:00, ?it/s]

✓ RECONSTRUCTION ET CORRECTION TERMINÉES !


In [ ]:
# 1. Force le démontage et remontage propre de Google Drive (résout le bug du cache Drive)
from google.colab import drive
import os
print("Rafraîchissement de la connexion Google Drive...")
drive.mount("/content/drive", force_remount=True)

# 2. Copier l'archive valide vers le disque local ultra-rapide de Colab
print("Copie de l'archive bdd100k_coco.zip en local...")
!cp /content/drive/MyDrive/datasets/bdd100k_coco.zip /content/

# 3. Décompresser proprement
print("Décompression du dataset d'entraînement...")
!rm -rf /content/bdd100k_coco
!unzip -q /content/bdd100k_coco.zip -d /content/
print("✓ Dataset local réinstallé avec succès dans : /content/bdd100k_coco")

Rafraîchissement de la connexion Google Drive...
Mounted at /content/drive
Copie de l'archive bdd100k_coco.zip en local...
cp: cannot stat '/content/drive/MyDrive/datasets/bdd100k_coco.zip': No such file or directory
Décompression du dataset d'entraînement...
unzip:  cannot find or open /content/bdd100k_coco.zip, /content/bdd100k_coco.zip.zip or /content/bdd100k_coco.zip.ZIP.
✓ Dataset local réinstallé avec succès dans : /content/bdd100k_coco


In [ ]:
import os
import json
import glob
from tqdm import tqdm

target_dir = "/content/bdd100k_coco"
zip_drive_path = "/content/drive/MyDrive/datasets/bdd100k_coco.zip"
zip_local_path = "/content/bdd100k_coco.zip"

# ==========================================================
# 1. REMONTAGE DE SÉCURITÉ DE GOOGLE DRIVE
# ==========================================================
from google.colab import drive
if not os.path.exists('/content/drive'):
    print("Connexion à Google Drive...")
    drive.mount('/content/drive')
else:
    print("Google Drive est déjà connecté.")

# ==========================================================
# 2. STRATÉGIE A : DÉCOMPRESSION DU ZIP DEPUIS LE DRIVE
# ==========================================================
if os.path.exists(zip_drive_path):
    print(f"✓ Archive ZIP détectée sur le Drive ({os.path.getsize(zip_drive_path) / (1024**3):.1f} Go).")
    print("Copie de l'archive vers le disque rapide local...")
    !cp {zip_drive_path} {zip_local_path}

    print("Décompression du dataset en cours...")
    !rm -rf {target_dir}
    !unzip -q {zip_local_path} -d /content/

    # Nettoyage du zip temporaire pour libérer de l'espace sur la VM
    if os.path.exists(zip_local_path):
        os.remove(zip_local_path)

# ==========================================================
# 3. STRATÉGIE B : RECONSTRUCTION PAR SYMLINKS (FALBACK)
# ==========================================================
if not os.path.exists(os.path.join(target_dir, "annotations/instances_train2017.json")):
    print("⚠ Le ZIP n'a pas pu être extrait ou est incomplet.")
    print("Bascule automatique sur la méthode de reconstruction par liens symboliques...")

    # Détection des sources d'images
    local_img_train = "/content/bdd100k_local/bdd100k/bdd100k/images/100k/train"
    local_img_val = "/content/bdd100k_local/bdd100k/bdd100k/images/100k/val"
    drive_img_train = "/content/drive/MyDrive/datasets/bdd100k/images/100k/train"
    drive_img_val = "/content/drive/MyDrive/datasets/bdd100k/images/100k/val"

    if os.path.exists(local_img_train):
        img_train_src, img_val_src = local_img_train, local_img_val
    elif os.path.exists(drive_img_train):
        img_train_src, img_val_src = drive_img_train, drive_img_val
    else:
        raise FileNotFoundError("Erreur critique : Aucune source d'images BDD100k détectée (ni locale, ni Drive).")

    # Détection des annotations
    local_anno_train = "/content/bdd100k_local/annotations/instances_train2017.json"
    local_anno_val = "/content/bdd100k_local/annotations/instances_val2017.json"
    drive_anno_train = "/content/drive/MyDrive/datasets/bdd100k/annotations/instances_train2017.json"
    drive_anno_val = "/content/drive/MyDrive/datasets/bdd100k/annotations/instances_val2017.json"

    if os.path.exists(local_anno_train):
        anno_train_src, anno_val_src = local_anno_train, local_anno_val
    elif os.path.exists(drive_anno_train):
        anno_train_src, anno_val_src = drive_anno_train, drive_anno_val
    else:
        raise FileNotFoundError("Erreur critique : Aucun fichier d'annotations JSON détecté.")

    # Reconstruction de la structure
    !rm -rf {target_dir}
    os.makedirs(os.path.join(target_dir, "annotations"), exist_ok=True)
    os.makedirs(os.path.join(target_dir, "train2017"), exist_ok=True)
    os.makedirs(os.path.join(target_dir, "val2017"), exist_ok=True)

    os.symlink(anno_train_src, os.path.join(target_dir, "annotations/instances_train2017.json"))
    os.symlink(anno_val_src, os.path.join(target_dir, "annotations/instances_val2017.json"))

    print("Mise à plat des images d'entraînement (symlinks)...")
    train_jpgs = glob.glob(os.path.join(img_train_src, "**/*.jpg"), recursive=True)
    for filepath in tqdm(train_jpgs, desc="Train"):
        filename = os.path.basename(filepath)
        os.symlink(filepath, os.path.join(target_dir, "train2017", filename))

    print("Mise à plat des images de validation (symlinks)...")
    val_jpgs = glob.glob(os.path.join(img_val_src, "**/*.jpg"), recursive=True)
    for filepath in tqdm(val_jpgs, desc="Val"):
        filename = os.path.basename(filepath)
        os.symlink(filepath, os.path.join(target_dir, "val2017", filename))

# ==========================================================
# 4. SÉCURITÉ : NETTOYAGE DES JSON (ANTI-CRASH)
# ==========================================================
def prune_json(json_path, img_dir):
    print(f"\nNettoyage de sécurité sur {json_path}...")
    if not os.path.exists(json_path):
        print("  [X] Fichier JSON introuvable.")
        return

    with open(json_path, "r") as f:
        data = json.load(f)

    original_images = len(data["images"])
    existing_files = set(os.listdir(img_dir))

    valid_images = [img for img in data["images"] if img["file_name"] in existing_files]
    valid_ids = {img["id"] for img in valid_images}
    valid_annos = [ann for ann in data["annotations"] if ann["image_id"] in valid_ids]

    data["images"] = valid_images
    data["annotations"] = valid_annos

    with open(json_path, "w") as f:
        json.dump(data, f)
    print(f"  [✓] Pruning terminé : {len(valid_images)} / {original_images} images physiquement validées.")

prune_json(os.path.join(target_dir, "annotations/instances_train2017.json"), os.path.join(target_dir, "train2017"))
prune_json(os.path.join(target_dir, "annotations/instances_val2017.json"), os.path.join(target_dir, "val2017"))

print("\n=======================================================")
print("✓ DATASET PRÊT ET VÉRIFIÉ À 100% !")
print("Vous pouvez maintenant lancer l'entraînement de YOLOX.")
print("=======================================================")

Google Drive est déjà connecté.
⚠ Le ZIP n'a pas pu être extrait ou est incomplet.
Bascule automatique sur la méthode de reconstruction par liens symboliques...
Mise à plat des images d'entraînement (symlinks)...


Train: 100%|██████████| 62538/62538 [00:03<00:00, 17301.54it/s]


Mise à plat des images de validation (symlinks)...


Val: 0it [00:00, ?it/s]



Nettoyage de sécurité sur /content/bdd100k_coco/annotations/instances_train2017.json...
  [✓] Pruning terminé : 62405 / 69863 images physiquement validées.

Nettoyage de sécurité sur /content/bdd100k_coco/annotations/instances_val2017.json...
  [✓] Pruning terminé : 0 / 10000 images physiquement validées.

✓ DATASET PRÊT ET VÉRIFIÉ À 100% !
Vous pouvez maintenant lancer l'entraînement de YOLOX.


In [ ]:
import os
import json

def prune_missing_images_from_json(json_path, img_dir):
    print(f"\nNettoyage de sécurité sur {json_path}...")
    if not os.path.exists(json_path):
        print("  [X] Fichier JSON introuvable.")
        return

    with open(json_path, "r") as f:
        data = json.load(f)

    original_image_count = len(data["images"])
    existing_filenames = set(os.listdir(img_dir))

    valid_images = []
    valid_image_ids = set()
    for img in data["images"]:
        if img["file_name"] in existing_filenames:
            valid_images.append(img)
            valid_image_ids.add(img["id"])

    valid_annotations = []
    for ann in data["annotations"]:
        if ann["image_id"] in valid_image_ids:
            valid_annotations.append(ann)

    data["images"] = valid_images
    data["annotations"] = valid_annotations

    with open(json_path, "w") as f:
        json.dump(data, f)
    print(f"  [✓] Pruning terminé : {len(valid_images)} / {original_image_count} images physiquement validées.")

# Exécuter le nettoyage sur le train et le val local
prune_missing_images_from_json(
    "/content/bdd100k_coco/annotations/instances_train2017.json",
    "/content/bdd100k_coco/train2017"
)
prune_missing_images_from_json(
    "/content/bdd100k_coco/annotations/instances_val2017.json",
    "/content/bdd100k_coco/val2017"
)


Nettoyage de sécurité sur /content/bdd100k_coco/annotations/instances_train2017.json...
  [✓] Pruning terminé : 62405 / 62405 images physiquement validées.

Nettoyage de sécurité sur /content/bdd100k_coco/annotations/instances_val2017.json...
  [✓] Pruning terminé : 0 / 0 images physiquement validées.


In [ ]:
# 1. S'assurer que les poids sont téléchargés à l'emplacement absolu racine
!wget -q https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth -O /content/yolox_nano.pth

# 2. Lancer l'entraînement avec le chemin d'accès absolu (-c /content/yolox_nano.pth)
!cd /content/YOLOX/YOLOX && python tools/train.py \
    -f exps/custom/yolox_nano_custom.py \
    -c /content/yolox_nano.pth \
    -d 1 \
    -b 16

2026-05-25 20:27:57.395436: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/content/YOLOX/YOLOX/yolox/core/trainer.py:47: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=args.fp16)
2026-05-25 20:28:01 | INFO     | yolox.core.trainer:132 - args: Namespace(experiment_name='yolox_nano_bdd100k', name=None, dist_backend='nccl', dist_url=None, batch_size=16, devices=1, exp_file='exps/custom/yolox_nano_custom.py', resume=False, ckpt='/content/yolox_nano.pth', start_epoch=None, num_machines=1, machine_rank=0, fp16=False, cache=None, occupy=False, logger='tensorboard', opts=[])
2026-05-25 20:28:01 | INFO     | yolox.core.traine